# QSV2G — modular Kaggle experiment runner
This notebook installs the **same** modular package used by the CLI. CPU and Internet are required for first-time setup.
Quantum operations are software simulations; real ML-KEM/ML-DSA is used where supported.


In [ ]:
from pathlib import Path
import subprocess,sys,os
REPO="https://github.com/shanmukhdatta/QSV2G.git"
SOURCE_REVISION="refactor/modular-research-package"
STUDIES=["validation","security","performance","noise","availability","security-adversarial","reliability"]
FULL=False  # Change only when you want the full research workloads.
ROOT=Path("/kaggle/working/QSV2G")
if not ROOT.exists(): subprocess.run(["git","clone",REPO,str(ROOT)],check=True)
subprocess.run(["git","checkout","--detach",SOURCE_REVISION],cwd=ROOT,check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",".[dev,pqc]"],cwd=ROOT,check=True)
print("Pinned revision:",SOURCE_REVISION)


In [ ]:
for study in STUDIES:
    command=[sys.executable,"scripts/run_experiment.py","--study",study]
    if FULL: command.append("--full")
    subprocess.run(command,cwd=ROOT,check=True)
print("Completed selected studies")


In [ ]:
import zipfile,hashlib,json
out=Path("/kaggle/working/qsv2g_modular_results.zip")
paths=sorted(p for p in (ROOT/"results/raw").rglob("*") if p.is_file())
hashes={str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}
with zipfile.ZipFile(out,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in paths:z.write(p,arcname=p.relative_to(ROOT))
    z.writestr("manifest_sha256.json",json.dumps(hashes,indent=2))
print(out,len(paths),"artifacts")
